In [ ]:
import numpy as onp
from pathlib import Path
import sys
_project_root: Path = Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[0]))
import src.symmetry as sym
from src.categorization import show_latex_bmatrix
# shift op
def initKshit(p):
  # I = onp.eye(3, dtype=complex)
  # this is X from the homework
  g123 = onp.array([
    [0, 0, 1],
    [1, 0, 0],
    [0, 1, 0],
  ], dtype=complex)
  C3 = sym.generated_group(
    [sym.AffineOperation(g123, label="(123)")],
    name="C3",
    max_order=3,
  )
  # for g in C3: print(show_latex_bmatrix(g.matrix.real))

  # print(C3.matrices)

  # for g in (I, g123, g123 @ g123): print(show_latex_bmatrix(g.real))
  ps = onp.sqrt(onp.array([1-p, p/2, p/2]))
  return ps * C3.matrices
  # K0 = onp.sqrt(1 - p) * I
  # K1 = onp.sqrt(p / 2) * g123
  # K2 = onp.sqrt(p / 2) * (g123 @ g123)
  # return (K0, K1, K2)

# clock op
def initKcock(p):
  ω = onp.exp(2j * onp.pi / 3)
  I = onp.eye(3, dtype=complex)
  Z = onp.diag([1, ω, ω**2])

  K0 = onp.sqrt(1 - p) * I
  K1 = onp.sqrt(p) * Z
  return (K0, K1)


def channel(ψ: onp.ndarray, OSR: tuple[onp.ndarray, ...]):
  ρ = onp.outer(ψ, ψ.conj())
  ρ1 = sum(K @ ρ @ K.conj().T for K in OSR)
  return ρ1, sum(K.conj().T @ K for K in OSR) # completenes


p = 2 / 3
shift_OSR = initKshit(p)
shift_ψ = onp.array([1, 0, 0], dtype=complex)
rho_shift, comp_shift = channel(shift_ψ, shift_OSR)

print("Shift channel rho' at p=2/3:")
print(onp.round(rho_shift, 4))

print("\nShift channel completeness:")
print(onp.round(comp_shift, 4))

p = 0.5
clock_OSR = initKcock(p)
clock_ψ = onp.ones(3, dtype=complex) / onp.sqrt(3)
rho_clock, comp_clock = channel(clock_ψ, clock_OSR)

print("\nClock channel rho' at p=1/2:")
print(onp.round(rho_clock, 4))

print("\nClock channel completeness:")
print(onp.round(comp_clock, 4))

print("\nOff-diagonal magnitude at p=1/2:")
print(abs(rho_clock[0, 1]))

Shift channel rho' at p=2/3:
[[0.33+0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.33+0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.33+0.j]]

Shift channel completeness:
[[1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j]]

Clock channel rho' at p=1/2:
[[0.33+0.j   0.08-0.14j 0.08+0.14j]
 [0.08+0.14j 0.33+0.j   0.08-0.14j]
 [0.08-0.14j 0.08+0.14j 0.33+0.j  ]]

Clock channel completeness:
[[1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j]]

Off-diagonal magnitude at p=1/2:
0.16666666666666677


/Users/jakeschaefer/Desktop/Research_Stuff/mat_sci/crystals/src/symmetry.py:505: ComplexWarning: Casting complex values to real discards the imaginary part
  return tuple(onp.round(data / tol).astype(onp.int64).tolist())
